# 01 | Database Health & Pipeline Freshness
Monitor core PostgreSQL database health, examine row counts across tables, verify connection reliability, and check data ingestion freshness.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Load and clean DATABASE_URL environment variable
DATABASE_URL = os.environ.get("DATABASE_URL", "postgresql://localhost/postgres")
if DATABASE_URL.startswith("postgres://"):
    DATABASE_URL = DATABASE_URL.replace("postgres://", "postgresql://", 1)

engine = create_engine(DATABASE_URL)

# 2. Define reusable sql() helper function
def sql(query_str, params=None):
    """Executes a SQL query and returns a pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query_str), conn, params=params)

print(f"Successfully connected to database engine: {engine.url.database}")

### Table Size & Row Count Audit
Quickly inspect the overall volume of records residing in our core pipeline tables (`leads_sandbox` and `ai_tasks`).

In [ ]:
# Query row counts for primary tables
counts_query = """
    SELECT 'leads_sandbox' AS table_name, COUNT(*) AS row_count FROM leads_sandbox
    UNION ALL
    SELECT 'ai_tasks' AS table_name, COUNT(*) AS row_count FROM ai_tasks;
"""
df_counts = sql(counts_query)
df_counts